In [ ]:
# Passo 0: Setup e Instalação
%pip install google-adk -q
%pip install litellm -q
%pip install groq -q

print("Instalação concluída.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Instalação concluída.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Passo 1: Importação de Bibliotecas
import os
import asyncio
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

load_dotenv()

print("Bibliotecas importadas.")

Bibliotecas importadas.


In [7]:
# Passo 2: Configuração de Chaves (Substitua pelas suas!)
print("API Keys Set:")
print(f"Google API Key set: {'Yes' if os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
print(f"Groq API Key set: {'Yes' if os.environ.get('GROQ_API_KEY') and os.environ['GROQ_API_KEY'] != 'YOUR_GROQ_API_KEY' else 'No (not using!)'}")

print("Ambiente de chaves configurado.")

API Keys Set:
Google API Key set: Yes
Groq API Key set: Yes
Ambiente de chaves configurado.


In [8]:
# Passo 3: Definição dos Modelos
# Gemini para a orquestração complexa
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

# Llama para tarefas específicas via Groq
MODEL_LLAMA_3_3_70B = "groq/llama-3.3-70b-versatile" 

print("Modelos definidos.")

Modelos definidos.


In [9]:
# Passo 4: Ferramenta de Consulta de Chamados (Tickets)
def check_ticket_status(ticket_id: str) -> dict:
    """Verifica o status de um chamado de TI (ticket).

    Args:
        ticket_id (str): O número ou código do chamado (ex: 'TCK-101').

    Returns:
        dict: O status e os detalhes do chamado.
    """
    print(f"---  Ferramenta: check_ticket_status chamada para o ticket: {ticket_id} ---")
    
    # Banco de dados simulado
    mock_db = {
        "tck-101": {"status": "Aberto", "description": "Problema de acesso à VPN.", "urgency": "Alta"},
        "tck-102": {"status": "Resolvido", "description": "Instalação do pacote Office.", "urgency": "Baixa"}
    }
    
    ticket_normalized = ticket_id.lower().strip()
    
    if ticket_normalized in mock_db:
        return {"success": True, "data": mock_db[ticket_normalized]}
    return {"success": False, "message": f"Chamado '{ticket_id}' não encontrado no sistema."}

In [10]:
# Passo 5: Função Auxiliar de Conversa e Sessão
session_service = InMemorySessionService()

async def call_agent_async(query: str, runner, user_id, session_id):
    """Envia a query ao agente e imprime a resposta final."""
    print(f"\n Usuário: {query}")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "O agente não produziu uma resposta."

    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            elif event.actions and event.actions.escalate:
                final_response_text = f"Erro reportado: {event.error_message}"
            break 
            
    print(f" Agente: {final_response_text}")

In [11]:
# Passo 6: Ferramentas de Hardware e Senha
def reset_password(username: str) -> str:
    """Gera um link de redefinição de senha para um usuário específico.
    
    Args:
        username (str): O login ou nome do usuário.
    """
    print(f"---  Ferramenta: reset_password acionada para: {username} ---")
    return f"Link de redefinição de senha enviado para o e-mail cadastrado de '{username}'. Validade: 15 minutos."

def diagnose_hardware(device_type: str) -> str:
    """Fornece um diagnóstico inicial para problemas de hardware.
    
    Args:
        device_type (str): O tipo de equipamento (ex: 'impressora', 'monitor', 'teclado').
    """
    print(f"---  Ferramenta: diagnose_hardware acionada para: {device_type} ---")
    device_lower = device_type.lower()
    
    if "impressora" in device_lower:
        return "Diagnóstico de Impressora: Verifique se há papel enroscado e se o cabo de rede/USB está bem conectado. Limpe o spooler de impressão."
    elif "monitor" in device_lower:
        return "Diagnóstico de Monitor: Certifique-se de que o cabo HDMI/DisplayPort está firme em ambas as pontas. Teste o monitor em outra tomada."
    else:
        return f"Guia genérico para {device_type}: Desligue o equipamento, aguarde 30 segundos, ligue novamente e verifique se as luzes indicadoras acendem."

In [12]:
# Passo 7: Subagentes de Senha e Hardware (Usando Llama via Groq)
try:
    password_agent = Agent(
        name="password_agent",
        model=LiteLlm(model=MODEL_LLAMA_3_3_70B), 
        description="Especialista em segurança. Lida EXCLUSIVAMENTE com solicitações de redefinição ou troca de senha dos usuários.",
        instruction="Você é o agente de senhas do Helpdesk. Sua única função é usar a ferramenta 'reset_password' quando um usuário pedir para trocar, resetar ou relatar esquecimento de senha. Não resolva outros problemas.",
        tools=[reset_password]
    )
    print(f" Subagente '{password_agent.name}' criado.")
except Exception as e:
    print(f" Erro no password_agent: {e}")

try:
    hardware_agent = Agent(
        name="hardware_agent",
        model=LiteLlm(model=MODEL_LLAMA_3_3_70B),
        description="Especialista físico. Lida com problemas de equipamentos tangíveis como impressoras, monitores, mouses e teclados.",
        instruction="Você é o técnico de hardware. Use a ferramenta 'diagnose_hardware' para ajudar usuários com equipamentos defeituosos. Extraia o nome do equipamento do texto do usuário e passe para a ferramenta. Seja prático.",
        tools=[diagnose_hardware]
    )
    print(f" Subagente '{hardware_agent.name}' criado.")
except Exception as e:
    print(f" Erro no hardware_agent: {e}")

 Subagente 'password_agent' criado.
 Subagente 'hardware_agent' criado.


In [13]:
# Passo 8: Agente Coordenador (Root Agent usando Gemini)
if 'password_agent' in globals() and 'hardware_agent' in globals():
    helpdesk_coordinator = Agent(
        name="helpdesk_coordinator",
        model=MODEL_GEMINI_2_5_FLASH, # Gemini para orquestração complexa
        description="O ponto de contato principal do usuário para suporte de TI. Pode ver tickets ou delegar tarefas técnicas.",
        instruction="""Você é o Coordenador de Suporte de TI (Nível 1).
        Regras de operação:
        1. Se o usuário quiser saber o status de um chamado/ticket específico, use a ferramenta 'check_ticket_status'.
        2. Se o usuário falar sobre esquecer, trocar ou resetar a SENHA, delegue imediatamente para o 'password_agent'.
        3. Se o usuário relatar problemas físicos com equipamentos (impressora quebrou, monitor não liga, hardware), delegue para o 'hardware_agent'.
        4. Para assuntos fora do escopo de TI, informe educadamente que você é um agente de suporte técnico.
        """,
        tools=[check_ticket_status],
        sub_agents=[password_agent, hardware_agent]
    )
    print(f" Agente Coordenador '{helpdesk_coordinator.name}' montou sua equipe com sucesso!")
else:
    print(" Falha: Os subagentes não foram criados corretamente na célula anterior.")

 Agente Coordenador 'helpdesk_coordinator' montou sua equipe com sucesso!


In [14]:
# Passo 9: Interagindo com o Sistema de Helpdesk
async def run_helpdesk_simulation():
    print("\n---  Iniciando Simulação de Atendimento Helpdesk ---")
    APP = "it_helpdesk_app"
    USER = "colaborador_01"
    SESSION = "sessao_matutina"
    
    await session_service.create_session(app_name=APP, user_id=USER, session_id=SESSION)
    
    runner = Runner(
        agent=helpdesk_coordinator,
        app_name=APP,
        session_service=session_service
    )

    # Teste 1: Acesso direto à ferramenta do Root
    await call_agent_async("Pode verificar como está o meu chamado TCK-101?", runner, USER, SESSION)
    
    # Teste 2: Delegação para o Subagente de Hardware
    await call_agent_async("A impressora do setor financeiro está mastigando o papel e não imprime nada.", runner, USER, SESSION)
    
    # Teste 3: Delegação para o Subagente de Senha
    await call_agent_async("Esqueci minha senha de rede, meu usuário é 'danig', pode me ajudar?", runner, USER, SESSION)

# Executa a simulação
await run_helpdesk_simulation()


---  Iniciando Simulação de Atendimento Helpdesk ---

 Usuário: Pode verificar como está o meu chamado TCK-101?
---  Ferramenta: check_ticket_status chamada para o ticket: TCK-101 ---
 Agente: O status do seu chamado TCK-101 é: Aberto. A descrição é "Problema de acesso à VPN." e a urgência é Alta.

 Usuário: A impressora do setor financeiro está mastigando o papel e não imprime nada.
---  Ferramenta: diagnose_hardware acionada para: impressora ---
 Agente: Diagnóstico de Impressora: Verifique se há papel enroscado e se o cabo de rede/USB está bem conectado. Limpe o spooler de impressão.

 Usuário: Esqueci minha senha de rede, meu usuário é 'danig', pode me ajudar?
---  Ferramenta: reset_password acionada para: danig ---
 Agente: Link de redefinição de senha enviado para o e-mail cadastrado de 'danig'. Validade: 15 minutos.
